# 09 · Performance

**Level:** advanced · **Prerequisites:** [07](07_how_pizzetti_computes_a_light_curve.ipynb) · **Run time:** ~1 min

**Covers:** cost per sample in each regime, block skipping and the transit duty cycle, reusing models, fixed against free exponents, the grazing worst case, batman's `max_err` trade-off

Where does the time go, and how do you keep it low in a fit? Timings are for one thread; the ratios matter more than the absolute numbers.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
import batman
import pizzetti
from pizzetti import occult, series_cut

def best(f, repeat=7):
    f(); ts = []
    for _ in range(repeat):
        s = time.perf_counter(); f(); ts.append(time.perf_counter() - s)
    return min(ts)

LAWS = {"quadratic": [0.4, 0.25], "squareroot": [0.3, 0.2], "nonlinear": [0.5, 0.1, 0.1, -0.1], "power2": [0.7, 0.6]}

## 1. Cost per sample in each regime

In [2]:
rp = 0.1
print(f"{'law':11s} {'interior':>10s} {'limb':>10s} {'off star':>10s}   (ns per sample)")
for law, u in LAWS.items():
    zc = series_cut(rp, law, u)
    zi = np.linspace(0, zc, 100_000)
    zl = np.linspace(zc + 1e-9, 1 + rp - 1e-9, 20_000)
    zo = np.linspace(1 + rp, 3, 100_000)
    ns = [best(lambda: occult(zz, rp, u, law)) / zz.size * 1e9 for zz in (zi, zl, zo)]
    print(f"{law:11s} {ns[0]:10.1f} {ns[1]:10.1f} {ns[2]:10.1f}")

law           interior       limb   off star   (ns per sample)
quadratic          1.5       17.4        0.5
squareroot         3.7       40.6        1.1
nonlinear          3.5       52.7        1.2
power2             3.8       37.4        1.1


The interior series costs one or two nanoseconds; the limb, tens of nanoseconds (the exact solution for the
quadratic law, the interpolants for the others); samples off the star are skipped in blocks of 64.

## 2. The transit duty cycle

Long light curves are mostly out of transit. Blocks without any sample on the star cost almost nothing.

In [3]:
p = pizzetti.TransitParams()
p.t0, p.per, p.rp, p.a, p.inc, p.ecc, p.w, p.limb_dark, p.u = 0.0, 10.0, 0.05, 20.0, 89.7, 0.0, 90.0, "quadratic", [0.4, 0.25]
print(f"{'half-width [d]':>14s} {'in transit':>11s} {'ms per 10^5 points':>19s}")
for hw in (0.2, 0.5, 2.0, 5.0):
    t = np.linspace(-hw, hw, 100_000)
    m = pizzetti.TransitModel(p, t)
    frac = np.mean(m.ds < 1 + p.rp)
    print(f"{hw:14.1f} {frac:11.1%} {1e3 * best(lambda: m.light_curve(p)):19.3f}")

half-width [d]  in transit  ms per 10^5 points
           0.2       41.6%               0.186
           0.5       16.6%               0.109
           2.0        4.2%               0.069
           5.0        1.7%               0.063


## 3. In a fit: reuse the model

`light_curve(params)` recomputes the orbit (the separations z) only when an orbital parameter changed;
changing `rp` or the limb darkening costs only the flux. Building a new `TransitModel` for every call costs
about as much as recomputing the orbit, so it is mainly a matter of convenience; what matters is whether the
orbit changes.

In [4]:
t = np.linspace(-0.25, 0.25, 30_000)
m = pizzetti.TransitModel(p, t)
rng = np.random.default_rng(1)
def flux_only():                       # e.g. rp and u change, orbit fixed
    p.rp = 0.05 + 1e-6 * rng.random(); p.u = [0.4, 0.25 + 1e-3 * rng.random()]
    return m.light_curve(p)
def with_orbit():                      # inclination changes: separations recomputed
    p.inc = 89.7 + 1e-6 * rng.random()
    return m.light_curve(p)
def rebuild():
    p.inc = 89.7 + 1e-6 * rng.random()
    return pizzetti.TransitModel(p, t).light_curve(p)
for name, fn in (("flux only (orbit unchanged)", flux_only), ("orbit recomputed", with_orbit), ("new TransitModel every call", rebuild)):
    print(f"{name:30s} {1e3 * best(fn):.3f} ms per call")

flux only (orbit unchanged)    0.051 ms per call
orbit recomputed               0.104 ms per call
new TransitModel every call    0.105 ms per call


## 4. Fixed against free exponents

pizzetti caches everything that depends only on the exponents of the law (series tables, rule weights). With
the power-2 law and α free in a fit, every call has a new exponent and rebuilds them (tens of microseconds).

In [5]:
t = np.linspace(-0.25, 0.25, 3000)
p.limb_dark, p.u, p.rp = "power2", [0.7, 0.6], 0.1
m = pizzetti.TransitModel(p, t)
rng = np.random.default_rng(0)
def fixed():
    p.u = [0.7, 0.6]; return m.light_curve(p)
def free():
    p.u = [0.7, 0.5 + 0.2 * rng.random()]; return m.light_curve(p)
print(f"alpha fixed: {1e6 * best(fixed):.0f} us per call;  alpha free: {1e6 * best(free):.0f} us per call")

alpha fixed: 62 us per call;  alpha free: 106 us per call


## 5. The worst case: grazing transits

A grazing transit has no interior phase, so the fast series never applies. pizzetti still beats batman
there, by a smaller margin.

In [6]:
t = np.linspace(-0.25, 0.25, 10_000)
print(f"{'law':10s} {'b':>5s} {'batman [ms]':>12s} {'pizzetti [ms]':>14s} {'speed-up':>9s}")
for law in ("quadratic", "power2"):
    for b in (0.3, 0.9, 1.0):
        tm = {}
        for mod in (batman, pizzetti):
            q = mod.TransitParams()
            q.t0, q.per, q.rp, q.a, q.ecc, q.w, q.limb_dark, q.u = 0.0, 10.0, 0.1, 20.0, 0.0, 90.0, law, LAWS[law]
            q.inc = np.degrees(np.arccos(b / q.a))
            mm = mod.TransitModel(q, t)
            tm[mod.__name__] = best(lambda: mm.light_curve(q))
        print(f"{law:10s} {b:5.2f} {1e3 * tm['batman']:12.3f} {1e3 * tm['pizzetti']:14.3f} {tm['batman'] / tm['pizzetti']:8.1f}x")

law            b  batman [ms]  pizzetti [ms]  speed-up
quadratic   0.30        0.203          0.028      7.2x
quadratic   0.90        0.171          0.051      3.3x
quadratic   1.00        0.129          0.034      3.8x
power2      0.30        2.425          0.089     27.2x


power2      0.90        2.770          0.113     24.4x
power2      1.00        1.428          0.090     15.9x


## 6. batman's accuracy–speed trade-off

batman's numerical laws integrate to a tolerance `max_err` (ppm). Tightening it toward pizzetti's accuracy costs
time; pizzetti has no such knob because it is always at the level of rounding.

In [7]:
t = np.linspace(-0.12, 0.12, 4001)
q = pizzetti.TransitParams()
q.t0, q.per, q.rp, q.a, q.inc, q.ecc, q.w, q.limb_dark, q.u = 0.0, 3.5, 0.1, 9.0, 87.7, 0.0, 90.0, "power2", [0.7, 0.6]
ref = pizzetti.TransitModel(q, t, series_tol=0.0, limb="quadrature").light_curve(q)
rows = []
for me in (1.0, 0.1, 0.01):
    qb = batman.TransitParams()
    for k in vars(q): setattr(qb, k, getattr(q, k))
    mb = batman.TransitModel(qb, t, max_err=me)
    rows.append((f"batman max_err={me}", best(lambda: mb.light_curve(qb)), np.max(np.abs(mb.light_curve(qb) - ref))))
mp_ = pizzetti.TransitModel(q, t)
rows.append(("pizzetti", best(lambda: mp_.light_curve(q)), np.max(np.abs(mp_.light_curve(q) - ref))))
for name, tt, e in rows:
    print(f"{name:22s} {1e3 * tt:8.3f} ms   max error {e:.1e}")

batman max_err=1.0        1.589 ms   max error 1.0e-06
batman max_err=0.1        4.872 ms   max error 1.4e-07
batman max_err=0.01      14.883 ms   max error 4.9e-08
pizzetti                  0.074 ms   max error 7.1e-10


## What you learned

- Interior samples cost ~1 ns, limb samples tens of ns, off-star samples almost nothing.
- In a fit, keep one `TransitModel` and update `params`; fixed exponents are cached, free ones cost tens of µs per call.
- Grazing transits are the least favourable case and are still faster than batman.
- pizzetti's accuracy is not traded for speed.

**Next:** [10 · Linear basis light curves](10_linear_basis_light_curves.ipynb)